# STL EDA - 연간 트렌드 분석

**목적:** 논문 Figure 기준 냉난방 계량기들의 월별 트렌드를 연도별로 비교한다.

**대상 계량기:**
- Cooling system: H1.Z16(CM1), H1.Z11/Z12(CM2), H1.Z24/Z25(CM3), V.K21, H1.K11~K16, H2.K21
- Heating system: H1.Z20, H1.ZE20(CHP), H1.W11(Total), H1.W12(CHP)

**시간 축:** x축 = 1~12월, 연도별(2018~2023) 비교

**해상도:** 1h

In [2]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ems.db import load_env, connect
load_env()

In [3]:
# 대상 계량기 정의
COOLING_ELECTRIC = ['H1.Z16', 'H1.Z11', 'H1.Z12', 'H1.Z24', 'H1.Z25']
COOLING_THERMAL  = ['V.K21', 'H1.K11', 'H1.K12', 'H1.K14', 'H1.K15', 'H1.K16', 'H2.K21']
HEATING_ELECTRIC = ['H1.Z20', 'H1.ZE20']
HEATING_THERMAL  = ['H1.W11', 'H1.W12']

ALL_METERS = COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL

# 측정 항목 정의
ELECTRIC_MEASUREMENTS = ['P', 'I1', 'I2', 'I3', 'U1', 'U2', 'U3', 'PF', 'Q', 'W', 'f',
                          'P1', 'P2', 'P3', 'PF1', 'PF2', 'PF3', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out']
THERMAL_MEASUREMENTS  = ['P', 'Tdiff', 'Trl', 'Tvl', 'V', 'W', 'qv']

print(f'전체 계량기 수: {len(ALL_METERS)}')

전체 계량기 수: 16


In [4]:
def fetch_meter_data(meter_urn: str, start: str = '2018-01-01', end: str = '2024-01-01') -> pd.DataFrame:
    """계량기 전체 측정 항목을 DB에서 읽어 wide format으로 반환한다."""
    sql = """
        SELECT ts, measurement, value
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND ts >= %s
          AND ts < %s
        ORDER BY ts, measurement
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, start, end))
    
    df['ts'] = pd.to_datetime(df['ts'], utc=True)
    wide = df.pivot_table(index='ts', columns='measurement', values='value', aggfunc='first')
    wide.columns.name = None
    return wide

print('fetch_meter_data 함수 준비 완료')

fetch_meter_data 함수 준비 완료


In [5]:
def plot_annual_trend(meter_urn: str, df: pd.DataFrame, title_suffix: str = ''):
    """계량기의 측정 항목별 월별 평균을 연도별로 비교하는 Plotly subplot을 생성한다."""
    measurements = [c for c in df.columns]
    n_meas = len(measurements)
    
    # subplot 행/열 계산 (최대 4열)
    ncols = min(4, n_meas)
    nrows = (n_meas + ncols - 1) // ncols
    
    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=measurements,
        shared_xaxes=False,
    )
    
    years = sorted(df.index.year.unique())
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
        '#9467bd', '#8c564b', '#e377c2'
    ]
    
    for idx, meas in enumerate(measurements):
        row = idx // ncols + 1
        col = idx % ncols + 1
        
        if meas not in df.columns:
            continue
        
        for y_idx, year in enumerate(years):
            year_data = df[df.index.year == year][meas]
            monthly_mean = year_data.groupby(year_data.index.month).mean()
            
            if monthly_mean.empty:
                continue
            
            fig.add_trace(
                go.Scatter(
                    x=monthly_mean.index,
                    y=monthly_mean.values,
                    name=str(year),
                    legendgroup=str(year),
                    showlegend=(idx == 0),
                    line=dict(color=colors[y_idx % len(colors)]),
                    mode='lines+markers',
                    marker=dict(size=5),
                ),
                row=row, col=col
            )
        
        # x축 월 레이블
        fig.update_xaxes(
            tickvals=list(range(1, 13)),
            ticktext=['1월','2월','3월','4월','5월','6월',
                      '7월','8월','9월','10월','11월','12월'],
            row=row, col=col
        )
    
    fig.update_layout(
        title=f'{meter_urn} 연간 트렌드 (월별 평균) {title_suffix}',
        height=max(300, nrows * 250),
        width=1400,
        legend=dict(title='연도', orientation='v'),
        template='plotly_white',
    )
    
    return fig

print('plot_annual_trend 함수 준비 완료')

plot_annual_trend 함수 준비 완료


## 1. Cooling system - 전기 계량기

In [5]:
for meter in COOLING_ELECTRIC:
    print(f'조회 중: {meter}')
    df = fetch_meter_data(meter)
    print(f'  → {len(df)}행, 측정항목: {list(df.columns)}')
    fig = plot_annual_trend(meter, df, '(Cooling Electric)')
    fig.show()

조회 중: H1.Z16


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52584행, 측정항목: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']


조회 중: H1.Z11


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


...

/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52583행, 측정항목: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']


조회 중: H1.Z25


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52583행, 측정항목: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']


## 2. Cooling system - 열량계

In [6]:
for meter in COOLING_THERMAL:
    print(f'조회 중: {meter}')
    df = fetch_meter_data(meter)
    print(f'  → {len(df)}행, 측정항목: {list(df.columns)}')
    fig = plot_annual_trend(meter, df, '(Cooling Thermal)')
    fig.show()

조회 중: V.K21


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52583행, 측정항목: ['P', 'Tdiff', 'Trl', 'Tvl', 'V', 'W', 'qv']


조회 중: H1.K11


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


...

/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 44677행, 측정항목: ['P', 'Tdiff', 'Trl', 'Tvl', 'V', 'W', 'qv']


조회 중: H2.K21


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52583행, 측정항목: ['P', 'Tdiff', 'Trl', 'Tvl', 'V', 'W', 'qv']


## 3. Heating system - 전기 계량기

In [7]:
for meter in HEATING_ELECTRIC:
    print(f'조회 중: {meter}')
    df = fetch_meter_data(meter)
    print(f'  → {len(df)}행, 측정항목: {list(df.columns)}')
    fig = plot_annual_trend(meter, df, '(Heating Electric)')
    fig.show()

조회 중: H1.Z20


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52584행, 측정항목: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']


조회 중: H1.ZE20


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 9519행, 측정항목: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'U1', 'U2', 'U3', 'W', 'W1', 'W2', 'W3', 'W_in', 'W_out', 'f']


## 4. Heating system - 열량계

In [8]:
for meter in HEATING_THERMAL:
    print(f'조회 중: {meter}')
    df = fetch_meter_data(meter)
    print(f'  → {len(df)}행, 측정항목: {list(df.columns)}')
    fig = plot_annual_trend(meter, df, '(Heating Thermal)')
    fig.show()

조회 중: H1.W11


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52583행, 측정항목: ['P', 'Tdiff', 'Trl', 'Tvl', 'V', 'W', 'qv']


조회 중: H1.W12


/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52583행, 측정항목: ['P', 'Tdiff', 'Trl', 'Tvl', 'V', 'W', 'qv']


In [11]:
for meter in COOLING_ELECTRIC:
    print(f'조회 중: {meter}')
    df = fetch_meter_data(meter)
    print(f'  → {len(df)}행, 측정항목: {list(df.columns)}')
    fig = plot_annual_trend(meter, df, '(Cooling Electric)')
    fig.write_image(f'outputs/figures/stl_eda/{meter}_annual_trend.png', width=1400, height=800)
    print(f'{meter} 저장 완료')

조회 중: H1.Z16


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


  → 52584행, 측정항목: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [14]:
import os
save_dir = ROOT / 'outputs/tables/stl_eda'
os.makedirs(save_dir, exist_ok=True)

for meter in COOLING_ELECTRIC:
    df = fetch_meter_data(meter)
    monthly = df.copy()
    monthly['year'] = monthly.index.year
    monthly['month'] = monthly.index.month
    monthly_mean = monthly.groupby(['year', 'month']).mean()
    monthly_mean.to_csv(save_dir / f'{meter}_monthly_mean.csv')
    print(f'{meter} 저장 완료')

/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z16 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z11 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z12 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z24 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z25 저장 완료


In [15]:
for meter in COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL:
    df = fetch_meter_data(meter)
    monthly = df.copy()
    monthly['year'] = monthly.index.year
    monthly['month'] = monthly.index.month
    monthly_mean = monthly.groupby(['year', 'month']).mean()
    monthly_mean.to_csv(save_dir / f'{meter}_monthly_mean.csv')
    print(f'{meter} 저장 완료')
    

/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


V.K21 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.K11 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


...

H1.ZE20 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.W11 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.W12 저장 완료


In [16]:
import os
os.makedirs(str(ROOT / 'outputs/figures/stl_eda'), exist_ok=True)

for meter in COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL:
    df = fetch_meter_data(meter)
    fig = plot_annual_trend(meter, df, '')
    fig.write_html(str(ROOT / f'outputs/figures/stl_eda/{meter}_annual_trend.html'))
    print(f'{meter} 저장 완료')

/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z16 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.Z11 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


...

H1.ZE20 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.W11 저장 완료


/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


H1.W12 저장 완료


In [17]:
import os
os.makedirs(str(ROOT / 'outputs/figures/stl_eda'), exist_ok=True)

for meter in COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL:
    df = fetch_meter_data(meter)
    fig = plot_annual_trend(meter, df, '')
    fig.write_image(str(ROOT / f'outputs/figures/stl_eda/{meter}_annual_trend.png'), width=1400, height=800)
    print(f'{meter} 저장 완료')

/tmp/ipykernel_9159/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [7]:
def plot_annual_trend_from_csv(meter_urn: str, df: pd.DataFrame, title_suffix: str = ''):
    measurements = [c for c in df.columns if c not in ['year', 'month']]
    n_meas = len(measurements)
    ncols = min(4, n_meas)
    nrows = (n_meas + ncols - 1) // ncols

    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=measurements)

    years = sorted(df['year'].unique())
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

    for idx, meas in enumerate(measurements):
        row = idx // ncols + 1
        col = idx % ncols + 1

        for y_idx, year in enumerate(years):
            year_data = df[df['year'] == year]
            fig.add_trace(
                go.Scatter(
                    x=year_data['month'],
                    y=year_data[meas],
                    name=str(year),
                    legendgroup=str(year),
                    showlegend=(idx == 0),
                    line=dict(color=colors[y_idx % len(colors)]),
                    mode='lines+markers',
                    marker=dict(size=5),
                ),
                row=row, col=col
            )
        fig.update_xaxes(
            tickvals=list(range(1, 13)),
            ticktext=['1월','2월','3월','4월','5월','6월','7월','8월','9월','10월','11월','12월'],
            row=row, col=col
        )

    fig.update_layout(
        title=f'{meter_urn} 연간 트렌드 (월별 평균) {title_suffix}',
        height=max(300, nrows * 250),
        width=1400,
        legend=dict(title='연도'),
        template='plotly_white',
    )
    return fig

In [9]:
import pandas as pd
from pathlib import Path

save_dir = ROOT / 'outputs/tables/stl_eda'

for meter in COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL:
    csv_path = save_dir / f'{meter}_monthly_mean.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        fig = plot_annual_trend_from_csv(meter, df, '')
        fig.write_image(str(ROOT / f'outputs/figures/stl_eda/{meter}_annual_trend.png'), width=1400, height=800)
        print(f'{meter} 저장 완료')
    else:
        print(f'{meter} CSV 없음')

H1.Z16 저장 완료
H1.Z11 저장 완료
H1.Z12 저장 완료
H1.Z24 저장 완료
H1.Z25 저장 완료
V.K21 저장 완료
H1.K11 저장 완료
H1.K12 저장 완료
H1.K14 저장 완료
H1.K15 저장 완료
H1.K16 저장 완료
H2.K21 저장 완료
H1.Z20 저장 완료
H1.ZE20 저장 완료
H1.W11 저장 완료
H1.W12 저장 완료


In [9]:
for meter in COOLING_ELECTRIC + COOLING_THERMAL + HEATING_ELECTRIC + HEATING_THERMAL:
    df = fetch_meter_data(meter)
    fig = plot_annual_trend(meter, df, '')
    fig.write_image(str(ROOT / f'outputs/figures/stl_eda/{meter}_annual_trend.png'), width=1400, height=800)
    print(f'{meter} 저장 완료')

/tmp/ipykernel_27081/1802778514.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, start, end))


RuntimeError: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome

